# 04 - RAG Evaluation

Evaluate end-to-end answers with an LLM-as-a-judge, and compare two system prompts (`INSTRUCTIONS` vs `INSTRUCTIONS_MIXOLOGIST`).

In [ ]:
import sys
sys.path.append('../cocktail_assistant')

from dotenv import load_dotenv
load_dotenv('../.env')

In [ ]:
import pandas as pd
from openai import OpenAI
from ingest import load_data, build_index, build_vector_index
from embedder import Embedder
from rag_helper import RAGHybrid, INSTRUCTIONS, INSTRUCTIONS_MIXOLOGIST
from judge import evaluate_relevance

client = OpenAI()

In [ ]:
documents = load_data('../data/cocktails.csv')
ground_truth = pd.read_csv('../data/ground_truth.csv').to_dict('records')

text_index = build_index(documents)
embedder = Embedder('../models/Xenova/all-MiniLM-L6-v2')
vector_index = build_vector_index(documents, embedder)

## Build a RAG assistant per prompt variant

In [ ]:
def make_assistant(instructions):
    return RAGHybrid(
        index=text_index,
        vector_index=vector_index,
        embedder=embedder,
        llm_client=client,
        instructions=instructions,
    )


assistants = {
    "default": make_assistant(INSTRUCTIONS),
    "mixologist": make_assistant(INSTRUCTIONS_MIXOLOGIST),
}

## Judge relevance on a sample of questions

In [ ]:
sample = ground_truth[:25]


def evaluate_prompt(assistant):
    verdicts = []
    for row in sample:
        answer = assistant.rag(row["question"])
        relevance, explanation = evaluate_relevance(row["question"], answer, client)
        verdicts.append(relevance)
    return pd.Series(verdicts).value_counts(normalize=True)


report = {name: evaluate_prompt(a) for name, a in assistants.items()}
pd.DataFrame(report).fillna(0)

The prompt variant with the higher share of `RELEVANT` verdicts is the one shipped in `rag_helper.INSTRUCTIONS` (or swap in the mixologist prompt if it wins).